# CalibrateQwen 01: hard-label or teacher-completion SFT
We run one structured-output SFT variant and retain the Tinker checkpoint log for evaluation.

In [1]:
from pathlib import Path
REPO_ROOT = Path('/content/AutoRegressive-Bhasha')
if not REPO_ROOT.exists():
    !git clone https://github.com/ritwikraha/AutoRegressive-Bhasha.git /content/AutoRegressive-Bhasha
%cd /content/AutoRegressive-Bhasha/calibrate_qwen
!pip install -q -r requirements.txt

Cloning into '/content/AutoRegressive-Bhasha'...
remote: Enumerating objects: 176, done.
remote: Counting objects: 100% (176/176), done.
remote: Compressing objects: 100% (138/138), done.
remote: Total 176 (delta 77), reused 112 (delta 36), pack-reused 0 (from 0)
Receiving objects: 100% (176/176), 1.89 MiB | 4.91 MiB/s, done.
Resolving deltas: 100% (77/77), done.
/content/AutoRegressive-Bhasha/calibrate_qwen
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.2/249.2 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 51.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 78.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 111.9 MB/s eta 0:00:00


In [2]:
import os
from google.colab import userdata
from huggingface_hub import login

os.environ['TINKER_API_KEY'] = userdata.get('TINKER_API_KEY')

hf_token = userdata.get("HF_WRITE_ACCESS").strip()
os.environ["HF_TOKEN"] = hf_token
login(token=hf_token, add_to_git_credential=False)

try:
    os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
except Exception:
    pass

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [3]:
METHOD = 'teacher'  # 'hard_label' or 'teacher'
CONFIDENCE_FORMAT = 'numeric'  # 'numeric', 'bucket', or 'implicit'
MAX_STEPS = None  # Set 3 for a paid pipeline smoke test
MODEL = 'Qwen/Qwen3.5-4B'
RUN_NAME = f'sft_{METHOD}_{CONFIDENCE_FORMAT}'
RUN_ROOT = Path('/content/calibrate_qwen_runs')
RUN_ROOT.mkdir(parents=True, exist_ok=True)

In [4]:
from training.prepare_training_data import prepare_training_file
from training.train_sft import SFTConfig, train_sft

conversation_file = f'artifacts/training/{METHOD}_{CONFIDENCE_FORMAT}.jsonl'
prepare_training_file(
    output_path=conversation_file,
    variant=METHOD,
    confidence_format=CONFIDENCE_FORMAT,
    abstention_threshold=0.6,
)
config = SFTConfig(
    conversation_file=conversation_file,
    log_path=str(RUN_ROOT / RUN_NAME),
    model_name=MODEL,
    batch_size=32,
    learning_rate=2e-4,
    lora_rank=32,
    max_steps=MAX_STEPS,
    wandb_project='calibrate-qwen' if os.environ.get('WANDB_API_KEY') else None,
    wandb_name=RUN_NAME,
)
config

README.md:   0%|          | 0.00/2.72k [00:00<?, ?B/s]

teacher/issue_phase1_teacher.parquet: reconstructing file:   0%|          |  0.00B / 6.13MB            

teacher/issue_phase1_teacher.parquet: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

SFTConfig(conversation_file='artifacts/training/teacher_numeric.jsonl', log_path='/content/calibrate_qwen_runs/sft_teacher_numeric', model_name='Qwen/Qwen3.5-4B', renderer_name='qwen3_5_disable_thinking', learning_rate=0.0002, batch_size=32, max_length=1024, num_epochs=1, lora_rank=32, test_size=100, save_every=20, eval_every=10, max_steps=None, wandb_project=None, wandb_name='sft_teacher_numeric', load_checkpoint_path=None)

In [5]:
await train_sft(config)

root:680 [INFO] Command line invocation: /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py -f /root/.local/share/jupyter/runtime/kernel-bbe66f5f-05ae-4abb-841a-35564c2de060.json
tinker_cookbook.utils.ml_log:618 [INFO] Logging to: /content/calibrate_qwen_runs/sft_teacher_numeric
tinker.lib.internal_client_holder:521 [WARNING] Your Tinker SDK version is outdated. Please upgrade to the latest version.
tinker.lib.public_interfaces.service_client:96 [INFO] ServiceClient initialized for session a85f6458-61a6-58a5-8f62-145d8d884867
tinker.lib.public_interfaces.service_client:203 [INFO] TrainingClient initialized for model a85f6458-61a6-58a5-8f62-145d8d884867:train:0


config.json:   0%|          | 0.00/3.16k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

...

tinker_cookbook.utils.ml_log:279 [INFO] 
              Step 54               
┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┓
┃ Metric               ┃ Value     ┃
┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━┩
│ epoch                │ 0         │
│ learning_rate        │ 0.000017  │
│ num_loss_tokens      │ 32.000000 │
│ num_sequences        │ 32        │
│ num_tokens           │ 11411     │
│ progress             │ 0.932203  │
│ time/finish_batch    │ 1.154450  │
│ time/get_batch       │ 0.067258  │
│ time/step            │ 1.142273  │
│ time/submit_batch    │ 0.070017  │
│ train_mean_bpb       │ 0.114068  │
│ train_mean_nll       │ 0.333907  │
└──────────────────────┴───────────┘
tinker_cookbook.supervised.train:419 [INFO] <|im_start|>system
Answer the multiple-choice question. Return one JSON object with exactly the keys "answer", "confidence", "justification", and "abstain". answer must be one uppercase option label from the prompt. confidence must be a number between 0 and 1. justification must be one

tinker_cookbook.utils.ml_log:279 [INFO] 
              Step 55               
┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┓
┃ Metric               ┃ Value     ┃
┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━┩
│ epoch                │ 0         │
│ learning_rate        │ 0.000014  │
│ num_loss_tokens      │ 32.000000 │
│ num_sequences        │ 32        │
│ num_tokens           │ 11499     │
│ progress             │ 0.949153  │
│ time/finish_batch    │ 1.045726  │
│ time/get_batch       │ 0.065580  │
│ time/step            │ 1.033484  │
│ time/submit_batch    │ 0.067948  │
│ train_mean_bpb       │ 0.119442  │
│ train_mean_nll       │ 0.354241  │
└──────────────────────┴───────────┘
tinker_cookbook.supervised.train:419 [INFO] <|im_start|>system
Answer the multiple-choice question. Return one JSON object with exactly the keys "answer", "confidence", "justification", and "abstain". answer must be one uppercase option label from the prompt. confidence must be a number between 0 and 1. justification must be one

tinker_cookbook.utils.ml_log:279 [INFO] 
              Step 56               
┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┓
┃ Metric               ┃ Value     ┃
┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━┩
│ epoch                │ 0         │
│ learning_rate        │ 0.000010  │
│ num_loss_tokens      │ 32.000000 │
│ num_sequences        │ 32        │
│ num_tokens           │ 12092     │
│ progress             │ 0.966102  │
│ time/finish_batch    │ 0.977828  │
│ time/get_batch       │ 0.067723  │
│ time/step            │ 0.965238  │
│ time/submit_batch    │ 0.070181  │
│ train_mean_bpb       │ 0.114972  │
│ train_mean_nll       │ 0.337189  │
└──────────────────────┴───────────┘
tinker_cookbook.supervised.train:419 [INFO] <|im_start|>system
Answer the multiple-choice question. Return one JSON object with exactly the keys "answer", "confidence", "justification", and "abstain". answer must be one uppercase option label from the prompt. confidence must be a number between 0 and 1. justification must be one

tinker_cookbook.utils.ml_log:279 [INFO] 
              Step 57               
┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┓
┃ Metric               ┃ Value     ┃
┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━┩
│ epoch                │ 0         │
│ learning_rate        │ 0.000007  │
│ num_loss_tokens      │ 32.000000 │
│ num_sequences        │ 32        │
│ num_tokens           │ 10620     │
│ progress             │ 0.983051  │
│ time/finish_batch    │ 1.286494  │
│ time/get_batch       │ 0.063698  │
│ time/step            │ 1.274679  │
│ time/submit_batch    │ 0.065767  │
│ train_mean_bpb       │ 0.113511  │
│ train_mean_nll       │ 0.335269  │
└──────────────────────┴───────────┘
tinker_cookbook.utils.ml_log:206 [INFO] Wrote metrics to /content/calibrate_qwen_runs/sft_teacher_numeric/metrics.jsonl


tinker_cookbook.utils.ml_log:279 [INFO] 
              Step 58               
┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┓
┃ Metric               ┃ Value     ┃
┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━┩
│ epoch                │ 0         │
│ learning_rate        │ 0.000003  │
│ num_loss_tokens      │ 32.000000 │
│ num_sequences        │ 32        │
│ num_tokens           │ 8017      │
│ progress             │ 1.000000  │
│ time/finish_batch    │ 1.151115  │
│ time/get_batch       │ 0.053308  │
│ time/step            │ 1.140508  │
│ time/submit_batch    │ 0.055550  │
│ train_mean_bpb       │ 0.138454  │
│ train_mean_nll       │ 0.399361  │
└──────────────────────┴───────────┘
tinker_cookbook.checkpoint_utils:467 [INFO] Saved checkpoints: {'state_path': 'tinker://a85f6458-61a6-58a5-8f62-145d8d884867:train:0/weights/final', 'sampler_path': 'tinker://a85f6458-61a6-58a5-8f62-145d8d884867:train:0/sampler_weights/final'}
tinker_cookbook.supervised.train:584 [INFO] Training completed successfully


In [6]:
checkpoint_log = Path(config.log_path) / 'checkpoints.jsonl'
print(checkpoint_log.read_text() if checkpoint_log.exists() else 'Checkpoint log will appear after the first save.')

{"name": "000020", "batch": 20, "epoch": 0, "state_path": "tinker://a85f6458-61a6-58a5-8f62-145d8d884867:train:0/weights/000020", "sampler_path": "tinker://a85f6458-61a6-58a5-8f62-145d8d884867:train:0/sampler_weights/000020", "elapsed_tokens": 230667}
{"name": "000040", "batch": 40, "epoch": 0, "state_path": "tinker://a85f6458-61a6-58a5-8f62-145d8d884867:train:0/weights/000040", "sampler_path": "tinker://a85f6458-61a6-58a5-8f62-145d8d884867:train:0/sampler_weights/000040", "elapsed_tokens": 461157}
{"name": "final", "batch": 0, "epoch": 1, "state_path": "tinker://a85f6458-61a6-58a5-8f62-145d8d884867:train:0/weights/final", "sampler_path": "tinker://a85f6458-61a6-58a5-8f62-145d8d884867:train:0/sampler_weights/final", "elapsed_tokens": 656707}

